# 02 Tract table built from the raw Overture layers
 Everything downstream works per Census tract, so this notebook turns the raw layers into one row per scored tract and then proves the aggregation is right.

The aggregation is `eda/build_tract_table.py` :

| Layer | Rule | Why |
|---|---|---|
| tract area | geodesic area of the tract polygon | no projection distortion |
| buildings | a footprint belongs to the tract containing its **centroid**; area is geodesic m2 | each building counted once even when it straddles a tract line |
| roads, unfiltered roads, rail | segment **clipped** to the tract in lon/lat, length measured in EPSG:5070 (km) | a long road belongs partly to each tract it crosses |
| infrastructure | features intersecting the tract | stations and airports are points or small areas |
| places (POIs) | point-in-tract | a place is a point |

A tract with no feature of some kind gets NaN here. Turning those into true zeros is a decision, made in notebook 03.

In [1]:
from common import *
import duckdb
t = pd.read_parquet(OUT / "tract_overture.parquet")
print(t.shape, "| regions:", t.region.value_counts().to_dict())
groups = {p: [c for c in t.columns if c.startswith(p)] for p in ["bldg_", "road_", "ru_", "rail_", "infra_", "poi_"]}
for g, cols in groups.items(): print(f"{g:7s} {len(cols):2d}:", ", ".join(cols))
print("\nkey unique:", t.GEOID.is_unique)

(9794, 60) | regions: {'south-central-tx': 6003, 'maricopa-az': 1593, 'eastern-ok': 1192, 'northern-ca': 591, 'eastern-wa': 415}
bldg_   18: bldg_n, bldg_area_m2, bldg_area_median_m2, bldg_n_lt30m2, bldg_n_ge300m2, bldg_n_height, bldg_n_floors, bldg_n_class, bldg_n_subtype, bldg_n_name, bldg_n_osm, bldg_n_msft, bldg_n_google, bldg_n_esri, bldg_n_other, bldg_src_n, bldg_year_median, bldg_n_upd_ge2024
road_   11: road_km_motorway, road_km_trunk, road_km_primary, road_km_secondary, road_km_tertiary, road_km_residential, road_km_unclassified, road_km_all, road_km_hwy, road_n_pieces, road_n_hwy_pieces
ru_      6: ru_km_all, ru_km_service, ru_km_track, ru_km_nonmotor, ru_km_unknown, ru_n_pieces
rail_    2: rail_km, rail_n_pieces
infra_   4: infra_n, infra_n_transit, infra_n_airport, infra_n_busstop
poi_    16: poi_n, poi_n_conf05, poi_n_conf08, poi_conf_mean, poi_n_open, poi_n_closed, poi_n_nocat, poi_n_distinct_cat, poi_src_n, poi_n_fire, poi_n_ems, poi_n_school, poi_n_hospital, poi_n_fire_

## 1. Reconciliation with raw totals
Totals computed straight from the raw files (no tract involved) against the sum of the tract table. Buildings and places should agree almost exactly; the small gap is features whose centroid or location falls in no scored tract (the seven dropped water tracts) or on a shared tract edge (counted in both).

In [2]:
c = duckdb.connect(); c.sql("load spatial"); c.sql("set geometry_always_xy = true")
def raw_n(r, layer): return c.sql(f"select count(*) from read_parquet('{(REF/r/f'{r}-overture-{layer}.parquet').as_posix()}')").fetchall()[0][0]
rows = []
for r in REG:
    g = t[t.region == r]
    rows.append(dict(region=r,
        bldg_raw=raw_n(r, "buildings"), bldg_tracts=g.bldg_n.sum(),
        poi_raw=raw_n(r, "pois"), poi_tracts=g.poi_n.sum(),
        infra_raw=raw_n(r, "infrastructure"), infra_tracts=g.infra_n.sum()))
rc = pd.DataFrame(rows).set_index("region")
for k in ["bldg", "poi", "infra"]: rc[f"{k}_diff"] = rc[f"{k}_tracts"] - rc[f"{k}_raw"]
print(rc.astype(int).to_string())

                  bldg_raw  bldg_tracts  poi_raw  poi_tracts  infra_raw  infra_tracts  bldg_diff  poi_diff  infra_diff
region                                                                                                                
maricopa-az        2908224      2908224   300046      300049       9482          9509          0         3          27
northern-ca        1164724      1164724   116897      116901       3318          3327          0         4           9
eastern-ok         2551694      2551694   207369      207371       2726          2735          0         2           9
south-central-tx  11463801     11463801  1300334     1300388      13622         13737          0        54         115
eastern-wa         1081346      1081345    76491       76492       4131          4139         -1         1           8


**Takeaway:** Buildings reconcile exactly in four regions and are one short in Eastern WA (a single footprint whose centroid lies in no scored tract). Place counts in the tract table exceed the raw totals by 1 to 54 (at most 0.004% of Texas's 1.3 million), which is points lying exactly on a shared tract edge and counted in both tracts. Infrastructure exceeds raw by 8 to 115 features (0.8% in Texas) because lines and areas that cross a tract line are counted in each tract they touch. These gaps are negligible for counts of this size.

## 2. Road and rail length against raw segment length
 A gap of a fraction of a percent is projection distortion, not missing roads.

In [3]:
rows = []
for r in REG:
    g = t[t.region == r]
    for layer, col in [("roads", "road_km_all"), ("roads-unfiltered", "ru_km_all"), ("rail", "rail_km")]:
        f = (REF / r / f"{r}-overture-{layer}.parquet").as_posix()
        raw = c.sql(f"select sum(ST_Length_Spheroid(geometry))/1000 from read_parquet('{f}')").fetchall()[0][0]
        rows.append(dict(region=r, layer=layer, raw_km=raw, tract_km=g[col].sum(), pct_diff=100 * (g[col].sum() / raw - 1)))
lk = pd.DataFrame(rows)
print(lk.pivot(index="region", columns="layer", values="pct_diff").round(3).to_string())
print(lk[lk.layer == "roads"].set_index("region")[["raw_km", "tract_km"]].round(0).to_string())

layer              rail  roads  roads-unfiltered
region                                          
eastern-ok        0.114  1.534             1.070
eastern-wa        0.049  0.122             0.059
maricopa-az      -0.108  0.034             0.018
northern-ca       0.321  1.610             0.736
south-central-tx  0.461  0.959             0.533
                      raw_km    tract_km
region                                  
maricopa-az      123,477.000 123,519.000
northern-ca       76,219.000  77,447.000
eastern-ok       210,095.000 213,319.000
south-central-tx 570,819.000 576,292.000
eastern-wa        79,064.000  79,160.000


Summed tract lengths are within a small margin of the geodesic length of the raw segments. No region is short of road, so nothing was dropped in the join. The remaining difference is most likely scale distortion in the equal-area projection used for lengths; that cause was not tested here, but the size (under 2%) is small next to the tract-to-tract variation in later notebooks.

## 3. A second, independent count for a sample of tracts
Recount buildings, places and highway pieces for 40 random tracts with a different predicate (`ST_Within` on the centroid instead of `ST_Intersects`, spatial filter first) and compare with the table.

In [4]:
rng = np.random.default_rng(20261025)
rows = []
for r in REG:
    g = t[t.region == r].sample(8, random_state=1)
    for geoid in g.GEOID:
        tg = f"(select geometry g from read_parquet('{tracts_parquet('census-tracts', r).as_posix()}') where GEOID='{geoid}')"
        P = lambda l: f"read_parquet('{(REF/r/f'{r}-overture-{l}.parquet').as_posix()}')"
        nb = c.sql(f"select count(*) from {P('buildings')} b, {tg} x where ST_Within(ST_Centroid(b.geometry), x.g)").fetchall()[0][0]
        npoi = c.sql(f"select count(*) from {P('pois')} p, {tg} x where ST_Within(p.geometry, x.g)").fetchall()[0][0]
        nr = c.sql(f"select count(*) from {P('roads')} q, {tg} x where ST_Intersects(q.geometry, x.g)").fetchall()[0][0]
        rows.append(dict(region=r, GEOID=geoid, bldg=nb, poi=npoi, road_pieces=nr))
chk = pd.DataFrame(rows).merge(t[["GEOID", "bldg_n", "poi_n", "road_n_pieces"]], on="GEOID")
chk[["bldg_n", "poi_n", "road_n_pieces"]] = chk[["bldg_n", "poi_n", "road_n_pieces"]].fillna(0)
chk["d_bldg"] = chk.bldg - chk.bldg_n; chk["d_poi"] = chk.poi - chk.poi_n; chk["d_road"] = chk.road_pieces - chk.road_n_pieces
print(len(chk), "tracts checked | exact matches: buildings", int((chk.d_bldg == 0).sum()), "places", int((chk.d_poi == 0).sum()), "road pieces", int((chk.d_road == 0).sum()))
print(chk.loc[(chk.d_bldg != 0) | (chk.d_poi != 0) | (chk.d_road != 0), ["region", "GEOID", "bldg", "bldg_n", "poi", "poi_n", "road_pieces", "road_n_pieces"]].to_string())

40 tracts checked | exact matches: buildings 40 places 40 road pieces 40
Empty DataFrame
Columns: [region, GEOID, bldg, bldg_n, poi, poi_n, road_pieces, road_n_pieces]
Index: []


**Takeaway:** A second count with a different predicate (centroid `ST_Within` instead of `ST_Intersects`) on 40 random tracts, 8 per region, matches the table exactly for buildings, places and road pieces (40 of 40 each). Together with the totals above, the aggregation is verified.

## 4. Which tracts have no feature of each kind?
NaN rows are tracts the join found nothing for. These are real facts about Overture (no road, no place) except where the tract is tiny or water, which notebook 03 examines.

In [5]:
cnt = ["bldg_n", "road_n_pieces", "road_n_hwy_pieces", "ru_n_pieces", "rail_n_pieces", "infra_n", "poi_n", "poi_n_fire", "poi_n_ems", "poi_n_school"]
nan = t.assign(**{k: t[k].isna() | (t[k] == 0) for k in cnt}).groupby("region")[cnt].mean().loc[REG]
print((100 * nan).round(1).T.to_string())
print("\nrows where bldg, road and place counts are all missing/zero:", int(((t.bldg_n.fillna(0) == 0) & (t.road_n_pieces.fillna(0) == 0) & (t.poi_n.fillna(0) == 0)).sum()))

region             maricopa-az  northern-ca  eastern-ok  south-central-tx  eastern-wa
bldg_n                   0.000        0.000       0.000             0.000       0.000
road_n_pieces            0.000        0.000       0.000             0.000       0.000
road_n_hwy_pieces        2.100        1.200       2.300             1.100       1.200
ru_n_pieces              0.000        0.000       0.000             0.000       0.000
rail_n_pieces           76.100       52.800      43.100            59.500      37.600
infra_n                 29.600       26.600      51.200            59.400      19.000
poi_n                    0.000        0.000       0.000             0.000       0.000
poi_n_fire              76.600       64.100      63.800            71.200      67.200
poi_n_ems               95.900       94.900      91.400            92.000      90.800
poi_n_school            26.100       15.200      19.500            21.500      17.100

rows where bldg, road and place counts are all missin